# Arm B — Clinically Defined Subgroups (Sex-Stratified)

This notebook implements Arm B of the study: patients are partitioned into two clinically defined subgroups by sex, and a classifier is trained independently within each subgroup. Arm B addresses RQ1 by comparing sex-stratified modelling against the non-stratified baseline established in Arm A (`global_model_armA.ipynb`).

Sex is used as the stratification variable because cardiovascular disease can differ between males and females in risk-factor profiles and feature–outcome relationships, with sex-specific models reported in the literature to differ in predictive performance and important predictors (proposal, Section 3.3). As a binary variable, sex also produces exactly two subgroups, which helps keep each subgroup large enough for reliable training given the limited sample size.

Arm B reuses Arm A's dataset, feature definitions, preprocessing, classifiers, hyperparameter grids, nested cross-validation procedure, evaluation metrics, and outer fold partitions unchanged. The only methodological difference from Arm A is that a separate logistic regression and random forest are fitted for the male subgroup and for the female subgroup, instead of one model fitted on the full population. Predictions from both subgroup models are pooled back into a single population-level validation set for the primary comparison against Arm A, as required by the proposal (Section 3.5).

## 2. Imports and configuration

`RANDOM_STATE` is fixed for reproducibility, and `K_OUTER = K_INNER = 5`, for the same reasons documented in Arm A (a small dataset, balancing per-fold sample size against stability). These values match Arm A exactly so that the inner cross-validation procedure is identical across arms.

In [1]:
import os
import hashlib
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Configuration (must match Arm A)
RANDOM_STATE = 42
K_OUTER      = 5
K_INNER      = 5

# Path constants -- this notebook lives in notebooks/, so PROJECT_ROOT is
# one level up; all data and outputs are read/written relative to it.
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, "heart+disease")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
PREDICTIONS_DIR = os.path.join(RESULTS_DIR, "predictions")
DIAGNOSTICS_DIR = os.path.join(PROJECT_ROOT, "diagnostics")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
os.makedirs(DIAGNOSTICS_DIR, exist_ok=True)

## 3. Load dataset

Arm B uses the same cleaned Cleveland extract as Arm A. The cleaning procedure (missing-value handling, duplicate check, and the resulting 303 → 297 record count) is documented in `data_cleaning.ipynb`; it is not repeated here, since Arm B must operate on the identical dataset rather than an independently re-derived one.

In [2]:
df = pd.read_csv(os.path.join(DATA_DIR, "cleveland_clean.csv"))
print("Loaded shape:", df.shape)
assert df.shape[0] == 297, "Arm B expects the same 297-record cleaned dataset used in Arm A."
df.head()

Loaded shape: (297, 15)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,check
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,False
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,True
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,True
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,False
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,False


## 4. Target construction

The original Cleveland target, `num`, represents the presence/severity of heart disease. For binary classification, observations with `num = 0` are coded as class 0 (absence of disease), while observations with `num > 0` are coded as class 1 (presence of disease). The derived `check` column is excluded from the feature matrix together with `num`, for the same leakage reason documented in Arm A.

In [3]:
target_col = "num"

y = (df[target_col] > 0).astype(int)
X = df.drop(columns=[target_col, "check"], errors="ignore")

print("Samples:", len(df), " Features:", X.shape[1])
print("Class balance:")
print(y.value_counts().rename({0: "no disease", 1: "disease"}))
print("Positive rate: {:.3f}".format(y.mean()))

Samples: 297  Features: 13
Class balance:
num
no disease    160
disease       137
Name: count, dtype: int64
Positive rate: 0.461


## 5. Feature definition

Continuous features are standardised and nominal category codes are one-hot encoded, identical to Arm A. The passthrough set differs from Arm A's, however: **`sex` is excluded from the subgroup feature matrix.**

Arm A and Arm C use all 13 features, including `sex` as a passthrough column. Arm B's subgroup classifiers see only 12: `fbs`, `exang`, `ca`, plus the standardised continuous and one-hot nominal features. This is not an uncontrolled difference between arms -- it is a direct consequence of the stratification variable itself being one of the original 13 features. Once patients are split by sex, `sex` is constant within each subgroup (verified in code below, not assumed) and therefore carries zero information for that subgroup's model: it cannot appear as a meaningful term in a fitted logistic regression, and it can never be the feature a random forest split usefully chooses. The information `sex` carries is not lost -- it is fully encoded in *which* subgroup a patient's model is fitted on, which is a strictly stronger use of that information than including it as a constant column ever could be.

Keeping the degenerate column is not merely redundant, it is measurably harmful: `RandomForestClassifier` uses `max_features="sqrt"` by default, so each split samples a small subset of the encoded columns (roughly 4 of ~22) as split candidates; a constant column that gets sampled wastes that slot, since it can never produce a split, occasionally starving the tree of a chance to consider a real predictor at that node instead. See Section 12 for the measured effect on this dataset.

In [4]:
continuous  = ["age", "trestbps", "chol", "thalach", "oldpeak"]
nominal     = ["cp", "restecg", "slope", "thal"]

# `sex` is the stratification variable itself. Verify -- not assume -- that it
# is constant within each subgroup before dropping it from the subgroup
# feature matrix; see the markdown above for why a constant column is
# actively harmful to the random forest, not just uninformative.
for group_name, code in {"male": 1.0, "female": 0.0}.items():
    n_unique = X.loc[df["sex"] == code, "sex"].nunique()
    assert n_unique == 1, f"Expected sex to be constant within the {group_name} subgroup, got {n_unique} values."
print("Confirmed: sex is constant within each subgroup (male=1.0 only, female=0.0 only).")

passthrough = ["fbs", "exang", "ca"]  # `sex` removed -- see markdown above.

print("Continuous:", continuous)
print("Nominal (one-hot encoded):", nominal)
print("Passthrough (already binary/count; sex excluded within subgroups):", passthrough)

Confirmed: sex is constant within each subgroup (male=1.0 only, female=0.0 only).
Continuous: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Nominal (one-hot encoded): ['cp', 'restecg', 'slope', 'thal']
Passthrough (already binary/count; sex excluded within subgroups): ['fbs', 'exang', 'ca']


## 6. Sex-stratified exploratory summary

Arm B partitions patients into exactly two subgroups using `sex`: `sex = 1` (male) and `sex = 0` (female). No additional clinical subgroups are created. The summary below reports subgroup sizes and disease rates, confirming that both subgroups are large and contain both outcome classes, which is required for stratified cross-validation to be well-defined within each subgroup. Patients are not removed to equalise subgroup sizes.

In [5]:
sex_label = df["sex"].map({1.0: "male", 0.0: "female"})

summary = pd.DataFrame({
    "n": sex_label.value_counts(),
    "no_disease": df.loc[y == 0, "sex"].map({1.0: "male", 0.0: "female"}).value_counts(),
    "disease": df.loc[y == 1, "sex"].map({1.0: "male", 0.0: "female"}).value_counts(),
}).fillna(0).astype(int)
summary["disease_rate"] = (summary["disease"] / summary["n"]).round(3)
summary.loc["total"] = [len(df), int((y == 0).sum()), int((y == 1).sum()), round(y.mean(), 3)]
summary

,n,no_disease,disease,disease_rate
sex,,,,
male,201.0,89.0,112.0,0.557
female,96.0,71.0,25.0,0.260
total,297.0,160.0,137.0,0.461


## 7. Preprocessing

Preprocessing that estimates parameters from the data is kept inside a scikit-learn `Pipeline` together with each classifier, exactly as in Arm A. Because a separate pipeline is fitted per outer fold per sex subgroup (Section 10), the scaler and encoder only ever see the training portion of that subgroup, preventing leakage from validation data. The `ColumnTransformer` definition is identical to Arm A's.

In [6]:
# Fitted only inside the pipeline, on training-fold data (see Section 10), never on the full dataset.
preprocess = ColumnTransformer([
    ("num",  StandardScaler(),                       continuous),
    ("cat",  OneHotEncoder(handle_unknown="ignore"),  nominal),
    ("pass", "passthrough",                           passthrough),
])

## 8. Reuse Arm A's outer folds

The proposal requires identical outer cross-validation fold partitions across all arms (Section 3.5). Arm A already created and saved this partition to `fold_id.csv`; Arm B loads it directly rather than generating a new one, so that every patient sits in exactly the same outer fold in Arm A and Arm B. Sex-specific train/validation masks are then derived by combining the shared fold assignment with the `sex` variable. The check below confirms that every fold, for both sexes, contains enough patients and both outcome classes for nested cross-validation to be well-defined.

In [7]:
FOLD_FILE = os.path.join(PROJECT_ROOT, "fold_id.csv")
if not os.path.exists(FOLD_FILE):
    raise FileNotFoundError(
        f"{FOLD_FILE} not found. Arm B requires the outer fold partition created by "
        "global_model_armA.ipynb; run that notebook first."
    )

fold_id = pd.read_csv(FOLD_FILE)["fold"].to_numpy()
if len(fold_id) != len(df):
    raise ValueError(
        f"{FOLD_FILE} has {len(fold_id)} entries but the current dataset has {len(df)} rows."
    )
print(f"Loaded outer fold assignment from {FOLD_FILE} (shared with Arm A).")

# Fold-file integrity guard (Task 9 / Arm A Section 8): fail loudly if this
# fold_id.csv is not the one Arm A actually produced, rather than silently
# training on a different partition than Arm A and Arm C.
MANIFEST_FILE = os.path.join(PROJECT_ROOT, "run_manifest.json")
if not os.path.exists(MANIFEST_FILE):
    raise FileNotFoundError(
        f"{MANIFEST_FILE} not found. Arm B requires the manifest written by "
        "global_model_armA.ipynb; run that notebook first."
    )
with open(MANIFEST_FILE) as f:
    manifest = json.load(f)
fold_id_md5 = hashlib.md5(open(FOLD_FILE, "rb").read()).hexdigest()
assert fold_id_md5 == manifest["fold_id_md5"], (
    f"{FOLD_FILE} MD5 ({fold_id_md5}) does not match run_manifest.json "
    f"({manifest['fold_id_md5']}); it was regenerated or edited since Arm A ran. "
    "Re-run global_model_armA.ipynb and then this notebook, in that order."
)
print(f"fold_id.csv MD5 verified against run_manifest.json: {fold_id_md5}")

sex = df["sex"].to_numpy()  # 1 = male, 0 = female

for k in range(K_OUTER):
    train, validation = (fold_id != k), (fold_id == k)
    for group_name, code in {"male": 1, "female": 0}.items():
        y_train_g = y[train & (sex == code)]
        y_val_g = y[validation & (sex == code)]
        print(f"fold {k} {group_name:6s} | train n={len(y_train_g):3d} classes={y_train_g.value_counts().to_dict()}"
              f" | validation n={len(y_val_g):3d} classes={y_val_g.value_counts().to_dict()}")

Loaded outer fold assignment from /Users/faye/Desktop/FIT2082/Research_heartDisease/FIT2082_Research/fold_id.csv (shared with Arm A).
fold_id.csv MD5 verified against run_manifest.json: 4d54a69c577f498114826fb495d7eaa8
fold 0 male   | train n=157 classes={1: 90, 0: 67} | validation n= 44 classes={1: 22, 0: 22}
fold 0 female | train n= 80 classes={0: 61, 1: 19} | validation n= 16 classes={0: 10, 1: 6}
fold 1 male   | train n=160 classes={1: 89, 0: 71} | validation n= 41 classes={1: 23, 0: 18}
fold 1 female | train n= 77 classes={0: 57, 1: 20} | validation n= 19 classes={0: 14, 1: 5}
fold 2 male   | train n=162 classes={1: 89, 0: 73} | validation n= 39 classes={1: 23, 0: 16}
fold 2 female | train n= 76 classes={0: 55, 1: 21} | validation n= 20 classes={0: 16, 1: 4}
fold 3 male   | train n=159 classes={1: 90, 0: 69} | validation n= 42 classes={1: 22, 0: 20}
fold 3 female | train n= 79 classes={0: 59, 1: 20} | validation n= 17 classes={0: 12, 1: 5}
fold 4 male   | train n=166 classes={1: 9

## 9. Model definitions and hyperparameter grids

Exactly the same two classifiers and hyperparameter grids as Arm A. Male and female models are tuned from the same candidate hyperparameters and the same selection procedure as the global model; only the data used to fit them differs.

In [8]:
models = {
    "logreg": (
        Pipeline([("pre", preprocess),
                  ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))]),
        {"clf__C": [0.01, 0.1, 1, 10]},
    ),
    "rf": (
        Pipeline([("pre", preprocess),
                  ("clf", RandomForestClassifier(random_state=RANDOM_STATE))]),
        {"clf__n_estimators": [200, 400], "clf__max_depth": [None, 5, 10]},
    ),
}


## 10. Nested cross-validation within sex subgroups

For each model and each outer fold, a separate model is trained for the male subgroup and for the female subgroup. Within each subgroup, hyperparameters are selected using `GridSearchCV` with an inner stratified cross-validation on that subgroup's outer-training patients only; the outer validation fold is never used for hyperparameter selection, exactly as in Arm A. ROC-AUC is used as the tuning criterion (`scoring="roc_auc"`), matching Arm A, because it is threshold-independent and is one of the three metrics reported for all arms. The selected model is refit on the full outer-training portion of that subgroup and used to predict the corresponding validation patients. Predicted class labels use the same fixed probability threshold of 0.5 as Arm A, applied identically to both sexes and both models, so that thresholding is not a source of variation between arms.

Male and female models are fitted completely independently: each subgroup's inner cross-validation only ever sees that subgroup's data, so a male model can never be tuned or validated using female patients, and vice versa. Because the two subgroups differ in composition, the selected hyperparameters may differ between sexes; this is expected and is not itself evidence of a methodological inconsistency, since both subgroups draw from the same grids and selection metric.

For every outer fold, the male model's predictions on the male validation patients and the female model's predictions on the female validation patients are pooled into a single population-level validation set before computing accuracy, F1, and ROC-AUC. This pooled, fold-level metric — not an average of the two subgroups' separate scores — is the primary Arm B result, since it is what is directly comparable to Arm A's population-level evaluation.

In [9]:
fold_results = []       # pooled, population-level: primary result
sex_fold_results = []   # sex-specific: secondary, descriptive result
oof_rows = []
patient_id = df.index.to_numpy()
sex_groups = {"male": 1, "female": 0}

for name, (pipe, grid) in models.items():
    for k in range(K_OUTER):
        train, validation = (fold_id != k), (fold_id == k)

        pooled_y_true, pooled_proba, pooled_pred = [], [], []
        group_aucs, group_ns = [], []  # for the sample-size-weighted AUC, Section 13

        for group_name, sex_code in sex_groups.items():
            group_train = train & (sex == sex_code)
            group_validation = validation & (sex == sex_code)

            # Hyperparameter selection uses only this subgroup's outer-training
            # patients; the outer validation fold is not seen until scoring below.
            inner = StratifiedKFold(n_splits=K_INNER, shuffle=True, random_state=RANDOM_STATE)
            search = GridSearchCV(pipe, grid, cv=inner, scoring="roc_auc", n_jobs=-1)
            search.fit(X[group_train], y[group_train])

            best = search.best_estimator_
            proba = best.predict_proba(X[group_validation])[:, 1]
            # Predicted probability of the positive class (heart disease present).
            pred = (proba >= 0.5).astype(int)  # Fixed threshold, consistent with Arm A.

            y_val = y[group_validation].to_numpy()
            group_auc = roc_auc_score(y_val, proba) if len(np.unique(y_val)) > 1 else np.nan

            sex_fold_results.append({
                "model": name, "fold": k, "sex": group_name,
                "n": int(group_validation.sum()),
                "accuracy": accuracy_score(y_val, pred),
                "f1": f1_score(y_val, pred, zero_division=0),
                "roc_auc": group_auc,
                "best_params": search.best_params_,
            })
            group_aucs.append(group_auc)
            group_ns.append(len(y_val))

            for pid, yt, p, c in zip(patient_id[group_validation], y_val, proba, pred):
                oof_rows.append({
                    "patient_id": int(pid), "fold": int(k), "sex": group_name,
                    "y_true": int(yt), "model": name, "proba": float(p), "pred": int(c),
                })

            pooled_y_true.append(y_val)
            pooled_proba.append(proba)
            pooled_pred.append(pred)

        # Population-level pooled result for this outer fold: male and female
        # validation predictions are combined before scoring, not averaged,
        # so the pooled metric reflects the full validation fold at once.
        y_true_pooled = np.concatenate(pooled_y_true)
        proba_pooled = np.concatenate(pooled_proba)
        pred_pooled = np.concatenate(pooled_pred)

        # Sample-size-weighted average of the two subgroups' own AUCs -- an
        # alternative to pooling probabilities that avoids mixing the two
        # models' independent calibrations (Section 13).
        aucs_arr = np.array(group_aucs, dtype=float)
        ns_arr = np.array(group_ns, dtype=float)
        valid = ~np.isnan(aucs_arr)
        roc_auc_weighted = float(np.average(aucs_arr[valid], weights=ns_arr[valid])) if valid.any() else np.nan

        fold_results.append({
            "model": name,
            "fold": k,
            "accuracy": accuracy_score(y_true_pooled, pred_pooled),
            "f1": f1_score(y_true_pooled, pred_pooled, zero_division=0),
            "roc_auc": roc_auc_score(y_true_pooled, proba_pooled),
            "roc_auc_weighted_subgroups": roc_auc_weighted,
        })

fold_results_df = pd.DataFrame(fold_results)
sex_fold_results_df = pd.DataFrame(sex_fold_results)
print("Pooled nested cross-validation complete:", len(fold_results_df), "model x fold rows")
print("Sex-specific nested cross-validation complete:", len(sex_fold_results_df), "model x fold x sex rows")

Pooled nested cross-validation complete: 10 model x fold rows
Sex-specific nested cross-validation complete: 20 model x fold x sex rows


## 11. Fold-level results

The pooled, population-level fold results are the primary output of this notebook and are saved for comparison with Arm A. Sex-specific fold-level results are also shown as a secondary, descriptive breakdown, but are not the basis for the Arm A vs Arm B comparison.

In [10]:
fold_results_df.to_csv(os.path.join(RESULTS_DIR, "armB_fold_results.csv"), index=False)
print("Saved armB_fold_results.csv (pooled, primary)")
display_pooled = fold_results_df.round(3)
display_pooled

Saved armB_fold_results.csv (pooled, primary)


,model,fold,accuracy,f1,roc_auc,roc_auc_weighted_subgroups
0,logreg,0,0.817,0.792,0.901,0.953
1,logreg,1,0.783,0.764,0.875,0.878
2,logreg,2,0.746,0.727,0.866,0.816
3,logreg,3,0.797,0.760,0.899,0.892
4,logreg,4,0.780,0.723,0.905,0.863
5,rf,0,0.867,0.862,0.948,0.968
6,rf,1,0.783,0.772,0.901,0.887
7,rf,2,0.712,0.702,0.853,0.820
8,rf,3,0.763,0.720,0.892,0.882
9,rf,4,0.881,0.863,0.916,0.871


In [11]:
sex_fold_results_display = sex_fold_results_df.drop(columns=["best_params"]).round(3)
sex_fold_results_display

,model,fold,sex,n,accuracy,f1,roc_auc
0,logreg,0,male,44,0.886,0.894,0.961
1,logreg,0,female,16,0.625,0.000,0.933
2,logreg,1,male,41,0.756,0.783,0.855
3,logreg,1,female,19,0.842,0.667,0.929
4,logreg,2,male,39,0.667,0.723,0.793
5,logreg,2,female,20,0.900,0.750,0.859
6,logreg,3,male,42,0.738,0.732,0.855
7,logreg,3,female,17,0.941,0.889,0.983
8,logreg,4,male,35,0.771,0.810,0.906
9,logreg,4,female,24,0.792,0.000,0.800


In [12]:
best_params_export = sex_fold_results_df[["model", "fold", "sex", "n", "best_params"]]
best_params_export.to_csv(os.path.join(DIAGNOSTICS_DIR, "armB_best_params.csv"), index=False)
print("Saved armB_best_params.csv")
best_params_export.head(10)

Saved armB_best_params.csv


,model,fold,sex,n,best_params
0,logreg,0,male,44,{'clf__C': 0.1}
1,logreg,0,female,16,{'clf__C': 0.01}
2,logreg,1,male,41,{'clf__C': 0.1}
3,logreg,1,female,19,{'clf__C': 1}
4,logreg,2,male,39,{'clf__C': 0.1}
5,logreg,2,female,20,{'clf__C': 0.1}
6,logreg,3,male,42,{'clf__C': 0.1}
7,logreg,3,female,17,{'clf__C': 1}
8,logreg,4,male,35,{'clf__C': 0.1}
9,logreg,4,female,24,{'clf__C': 0.01}


## 12. Overall performance summary

Mean and standard deviation across the 5 outer folds are reported for each model, using the pooled population-level fold results, so that Arm B's primary comparison with Arm A rests on the same kind of summary statistic in both notebooks. The corresponding sex-specific summary (Male LR, Male RF, Female LR, Female RF) is reported separately below as a secondary, descriptive result.

In [13]:
results = (
    fold_results_df
    .groupby("model")[["accuracy", "f1", "roc_auc"]]
    .agg(["mean", "std"])
)
results.columns = ["accuracy_mean", "accuracy_std", "f1_mean", "f1_std", "rocauc_mean", "rocauc_std"]
results = results.reset_index()

for _, row in results.iterrows():
    print(f"{row['model']:7s} | ACC {row['accuracy_mean']:.3f} +/- {row['accuracy_std']:.3f}"
          f" | F1 {row['f1_mean']:.3f} +/- {row['f1_std']:.3f}"
          f" | AUC {row['rocauc_mean']:.3f} +/- {row['rocauc_std']:.3f}")

results.to_csv(os.path.join(RESULTS_DIR, "armB_results.csv"), index=False)
print("Saved armB_results.csv (pooled, primary)")
results.set_index("model").round(3)

logreg  | ACC 0.784 +/- 0.026 | F1 0.753 +/- 0.029 | AUC 0.889 +/- 0.018
rf      | ACC 0.801 +/- 0.072 | F1 0.784 +/- 0.076 | AUC 0.902 +/- 0.034
Saved armB_results.csv (pooled, primary)


,accuracy_mean,accuracy_std,f1_mean,f1_std,rocauc_mean,rocauc_std
model,,,,,,
logreg,0.784,0.026,0.753,0.029,0.889,0.018
rf,0.801,0.072,0.784,0.076,0.902,0.034


In [14]:
sex_results = (
    sex_fold_results_df
    .groupby(["sex", "model"])[["accuracy", "f1", "roc_auc"]]
    .agg(["mean", "std"])
)
sex_results.columns = ["accuracy_mean", "accuracy_std", "f1_mean", "f1_std", "rocauc_mean", "rocauc_std"]
sex_results = sex_results.reset_index()

for _, row in sex_results.iterrows():
    label = f"{row['sex']} {row['model']}"
    print(f"{label:13s} | ACC {row['accuracy_mean']:.3f} +/- {row['accuracy_std']:.3f}"
          f" | F1 {row['f1_mean']:.3f} +/- {row['f1_std']:.3f}"
          f" | AUC {row['rocauc_mean']:.3f} +/- {row['rocauc_std']:.3f}")

sex_results.to_csv(os.path.join(RESULTS_DIR, "armB_sex_specific_results.csv"), index=False)
print("Saved armB_sex_specific_results.csv (secondary, descriptive)")
sex_results.round(3)

female logreg | ACC 0.820 +/- 0.123 | F1 0.461 +/- 0.428 | AUC 0.901 +/- 0.072
female rf     | ACC 0.863 +/- 0.030 | F1 0.710 +/- 0.062 | AUC 0.934 +/- 0.070
male logreg   | ACC 0.764 +/- 0.079 | F1 0.788 +/- 0.069 | AUC 0.874 +/- 0.063
male rf       | ACC 0.772 +/- 0.109 | F1 0.799 +/- 0.090 | AUC 0.866 +/- 0.086
Saved armB_sex_specific_results.csv (secondary, descriptive)


,sex,model,accuracy_mean,accuracy_std,f1_mean,f1_std,rocauc_mean,rocauc_std
0,female,logreg,0.820,0.123,0.461,0.428,0.901,0.072
1,female,rf,0.863,0.030,0.710,0.062,0.934,0.070
2,male,logreg,0.764,0.079,0.788,0.069,0.874,0.063
3,male,rf,0.772,0.109,0.799,0.090,0.866,0.086


## 13. Subgroup-weighted AUC and the calibration-mixing caveat

Arm B's pooled ROC-AUC (Section 12) ranks all patients in a validation fold together, using probabilities from two independently fitted models -- one for the male subgroup, one for the female. AUC measures how well predicted probabilities rank positives above negatives; ranking *across* the male/female boundary partly reflects how the two models' probability scales happen to align with each other, not only how well either one discriminates within its own subgroup. Accuracy and F1 do not have this problem, since the 0.5 threshold is applied within each subgroup before the predictions are pooled.

As a check that avoids mixing the two calibrations, each subgroup's own AUC (already reported per fold in `armB_sex_specific_results.csv`) is combined into a single number via a sample-size-weighted average instead of by pooling probabilities. This weighted-average AUC is reported below alongside the pooled figure it is a check on. **Any cross-arm AUC comparison that includes Arm B (or Arm C) therefore carries a calibration-mixing component that Arm A's pooled AUC does not** -- where the pooled and weighted-average numbers diverge noticeably, the difference is coming from calibration, not discrimination.

In [15]:
pooled_auc_by_model = (
    fold_results_df.groupby("model")["roc_auc"]
    .agg(["mean", "std"]).rename(columns={"mean": "rocauc_pooled_mean", "std": "rocauc_pooled_std"})
)
weighted_auc_by_model = (
    fold_results_df.groupby("model")["roc_auc_weighted_subgroups"]
    .agg(["mean", "std"]).rename(columns={"mean": "rocauc_weighted_mean", "std": "rocauc_weighted_std"})
)
auc_comparison = pooled_auc_by_model.join(weighted_auc_by_model).round(3)
auc_comparison.to_csv(os.path.join(DIAGNOSTICS_DIR, "armB_auc_pooled_vs_weighted.csv"))
print("Saved armB_auc_pooled_vs_weighted.csv")
auc_comparison

Saved armB_auc_pooled_vs_weighted.csv


,rocauc_pooled_mean,rocauc_pooled_std,rocauc_weighted_mean,rocauc_weighted_std
model,,,,
logreg,0.889,0.018,0.880,0.050
rf,0.902,0.034,0.886,0.053


## 13. Out-of-fold predictions

For every patient, the out-of-fold prediction from the one outer fold in which they were held out is retained, together with the patient identifier, fold id, sex, true label, model name, predicted probability, and predicted class. This matches Arm A's out-of-fold prediction structure (`patient_id`, `fold`, `y_true`, `model`, `proba`, `pred`) with `sex` added, so that Arm A, Arm B, and Arm C can be compared using the same patient identifiers, fold assignment, and column layout. Because each patient belongs to exactly one sex, they receive exactly one out-of-fold prediction per model, from the subgroup model that held them out.

In [16]:
oof_df = pd.DataFrame(oof_rows)

# Verify exactly one OOF prediction per patient per model, and full coverage of all patients.
counts_per_model = oof_df.groupby("model")["patient_id"].nunique()
assert (counts_per_model == len(df)).all(), "Every patient must receive exactly one OOF prediction per model."
assert oof_df.groupby(["model", "patient_id"]).size().max() == 1, "Duplicate OOF prediction detected for a patient."

oof_df.to_csv(os.path.join(PREDICTIONS_DIR, "armB_predictions.csv"), index=False)
print("Saved armB_predictions.csv:", oof_df.shape)
oof_df.head()

Saved armB_predictions.csv: (594, 7)


,patient_id,fold,sex,y_true,model,proba,pred
0,1,0,male,1,logreg,0.968242,1
1,2,0,male,1,logreg,0.958246,1
2,8,0,male,1,logreg,0.856418,1
3,10,0,male,0,logreg,0.481014,0
4,12,0,male,1,logreg,0.623042,1


## 14. Diagnosing the female logistic regression's threshold collapse

The female subgroup's logistic regression F1 is unstable across folds (Section 12: `armB_sex_specific_results.csv`). This is demonstrated below rather than asserted: for each fold, the `C` selected by the inner grid search, the maximum predicted probability issued to any female validation patient, and the female subgroup's training positive rate are reported together -- the three quantities needed to distinguish a genuine class-imbalance/small-sample failure to learn from a regularisation-and-threshold interaction where the model learns the ranking correctly but never crosses 0.5.

In [17]:
female_logreg_C = (
    sex_fold_results_df[(sex_fold_results_df["model"] == "logreg") & (sex_fold_results_df["sex"] == "female")]
    .assign(C=lambda d: d["best_params"].apply(lambda p: p["clf__C"]))
    .set_index("fold")[["C"]]
)

female_logreg_oof = oof_df[(oof_df["model"] == "logreg") & (oof_df["sex"] == "female")]
max_proba_by_fold = female_logreg_oof.groupby("fold")["proba"].max().rename("max_proba_female_val")

female_train_rate_by_fold = pd.Series(
    {k: y[(fold_id != k) & (sex == 0)].mean() for k in range(K_OUTER)},
    name="female_train_positive_rate",
)

female_auc_by_fold = (
    sex_fold_results_df[(sex_fold_results_df["model"] == "logreg") & (sex_fold_results_df["sex"] == "female")]
    .set_index("fold")["roc_auc"].rename("female_roc_auc")
)

diagnosis = (
    female_logreg_C.join(max_proba_by_fold).join(female_train_rate_by_fold).join(female_auc_by_fold).round(3)
)
diagnosis.to_csv(os.path.join(DIAGNOSTICS_DIR, "armB_female_logreg_diagnosis.csv"))
print("Saved armB_female_logreg_diagnosis.csv")
diagnosis

Saved armB_female_logreg_diagnosis.csv


,C,max_proba_female_val,female_train_positive_rate,female_roc_auc
fold,,,,
0,0.01,0.381,0.238,0.933
1,1.00,0.991,0.260,0.929
2,0.10,0.941,0.276,0.859
3,1.00,0.933,0.253,0.983
4,0.01,0.494,0.278,0.800


**Conclusion.** The collapse happens in folds 0 and 4 -- two folds, not one -- and in both cases the inner grid search selected the strongest regularisation on offer, `C = 0.01`, while every other fold selected `C >= 0.1`. Heavy L2 shrinkage pulls the fitted coefficients toward the intercept, which sits near the female training positive rate (0.238-0.278 across folds); the resulting predicted probabilities never reach 0.5 as a result -- the maximum predicted probability among female validation patients was 0.381 in fold 0 and 0.494 in fold 4. Class imbalance is not the mechanism: fold 4's training positive rate (0.278) is the *highest* of all five folds, not the lowest, yet it is one of the two that collapsed. The model was not failing to learn, either -- female ROC-AUC stayed at 0.80-0.98 across all five folds, including 0.933 in fold 0 and 0.800 in fold 4, meaning it ranked patients correctly in both "collapsed" folds. This is a regularisation-and-threshold interaction, not a small-sample learning failure: the classifier learned a useful ranking, and a heavily shrunk, intercept-dominated model simply never crossed the fixed 0.5 cutoff.

## 16. Interpretation / notes

The primary comparison for RQ1 is Arm A's global model (`armA_results.csv`) against Arm B's pooled, sex-stratified model (`armB_results.csv`); both report accuracy, F1-score, and ROC-AUC as mean ± standard deviation over the same 5 outer folds. The sex-specific results (`armB_sex_specific_results.csv`) and the diagnosis in Section 14 explain *where* and *why* any difference originates; they are not themselves the basis for answering RQ1. Whether Arm B outperforms, underperforms, or performs similarly to Arm A is an empirical outcome of this comparison, checked for statistical distinguishability in `cross_arm_comparison.ipynb` rather than assumed here.

Arm B's subgroup classifiers use 12 features, not 13 (Section 5): `sex` is excluded from the subgroup feature matrix because it is constant within each subgroup once patients are split by it, and a constant column is not merely uninformative for `RandomForestClassifier` under `max_features="sqrt"` but measurably degrades it. On this run, dropping `sex` left logistic regression's pooled accuracy exactly unchanged (0.784 before and after -- an L2-regularised linear model assigns a near-zero coefficient to a column with zero within-subgroup variance whether or not it is present, so removing it changes nothing) and moved random forest's pooled accuracy from 0.794 to 0.801 (F1 0.773 to 0.784, ROC-AUC 0.898 to 0.902), consistent with a degenerate column occasionally consuming a `max_features="sqrt"` split-candidate slot without ever being able to produce a split.

For comparability with Arm C: the same `fold_id.csv` partition (now integrity-checked via `run_manifest.json`, Section 8) and the same classifiers/grids used here should be reused unchanged, so that Arm C's data-driven clusters differ from Arm A and Arm B only in how patients are grouped. `passthrough` is the one deliberate exception to "same features everywhere": Arm A and Arm C use `["sex", "fbs", "exang", "ca"]`, Arm B uses `["fbs", "exang", "ca"]`, for the reason given in Section 5.